# Campus GPT - VTU AI&ML assistant
Runtime > Change runtime type > **T4 GPU**. Run the cells top to bottom. Upload `campus-gpt.zip` in step 1 and your course files in step 3.

In [ ]:
# 1. Upload campus-gpt.zip, unzip, install
from google.colab import files
up = files.upload()  # choose campus-gpt.zip
!unzip -q -o campus-gpt.zip -d /content
%cd /content/campus-gpt
!pip install -q -r requirements-colab.txt
!python -m pytest -q  # sanity: should print 'passed'

## 2. Chat with the verified rules + AIML scheme (works with no documents)

In [ ]:
from campus_gpt.bot import CampusBot
bot = CampusBot()
for q in ['minimum CIE marks','AIML 7th sem subjects','convert 8.2 cgpa to percentage','who won the world cup']:
    print('Q:', q); print(bot.answer(q)); print('-'*60)

## 3. Add your course material (PDF / PPTX / DOCX)
Upload files into `data/raw/`. Name them like `BCS502-module-3-textbook.pdf` or `BAI701-module-4-textbook.pdf` so the subject and module are detected. Raw files stay in Colab and are not committed anywhere.

In [ ]:
import os
os.makedirs('data/raw', exist_ok=True)
up = files.upload()
for n, b in up.items():
    open(os.path.join('data/raw', n), 'wb').write(b)
!python -m campus_gpt.ingest.build --input data/raw --out data/chunks.jsonl

## 4. Launch the chat (base model + retrieval). Open the public link printed below.

In [ ]:
from campus_gpt.rag import Retriever
from campus_gpt.llm import load_generator
from campus_gpt.app import launch
bot = CampusBot(retriever=Retriever('data/chunks.jsonl'), generate=load_generator())
launch(bot, share=True)

## 5. Fine-tune once (QLoRA, ~minutes on a T4)
Builds the grounded cite-or-refuse dataset (held-out split is leak-free) and trains a LoRA adapter.

In [ ]:
!python -m train.make_dataset --out data --chunks data/chunks.jsonl
!python train/qlora.py --train data/sft_train.jsonl --out adapters/campus-gpt

## 6. Evaluate before vs after, then chat with the fine-tuned adapter
`eval/results.json` holds the numbers for the paper.

In [ ]:
!python -m eval.run_eval --out eval/results_base.json
!python -m eval.run_eval --adapter adapters/campus-gpt --out eval/results_finetuned.json
!cat eval/results_finetuned.json

In [ ]:
bot = CampusBot(retriever=Retriever('data/chunks.jsonl'), generate=load_generator('adapters/campus-gpt'))
launch(bot, share=True)

In [ ]:
# Download the adapter + results
!zip -qr adapter_and_results.zip adapters eval/results_base.json eval/results_finetuned.json
files.download('adapter_and_results.zip')